In [3]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

server_name = "localhost"
db_name = "everyloop3"

connection_string = (
    "DRIVER=ODBC Driver 18 for SQL Server;"
    f"SERVER={server_name};"
    f"DATABASE={db_name};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes"
)

url_string = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": connection_string}
)

engine = create_engine(url_string)

try:
    with engine.connect() as conn:
        print("Connected successfully using Windows Authentication")
except Exception as e:
    print("Connection failed")
    print(e)


Connected successfully using Windows Authentication


In [4]:
from sqlalchemy import text

query = text("SELECT TOP 5 Id, FirstName, LastName, Email FROM USERS ORDER BY FirstName, LastName")
type(query)

sqlalchemy.sql.elements.TextClause

In [5]:

with engine.connect() as conn:
    result = conn.execute(query) # ask the database a question

    print(result)#You get a container with the answer
    print(type(result))#You check what kind of container it is
    print(type(result.fetchall()))#You open the container and take out all the rows#You check what kind of thing you got

<class 'sqlalchemy.engine.cursor.CursorResult'>
<class 'list'>


In [6]:
with engine.connect() as conn:
    result = conn.execute(query)

    for row in result:
        print(f"{row.FirstName} {row.LastName}")

Alexander Dahl
Alvin Lindholm
Anders Hansson
Anne Åkerman
Annette Bergfalk


In [7]:
with engine.connect( ) as conn:
    result= conn.execute(query)
    for row in result:
        print(row)


('741109-2058', 'Alexander', 'Dahl', 'alexander.dahl@telia.se')
('530720-7675', 'Alvin', 'Lindholm', 'alvin.lindholm@gmail.com')
('820624-3075', 'Anders', 'Hansson', 'anders.hansson@hotmail.com')
('751123-9724', 'Anne', 'Åkerman', 'anne.akerman@hotmail.com')
('620925-4245', 'Annette', 'Bergfalk', 'annette.bergfalk@telia.se')


In [8]:
with engine.connect() as conn:
    result = conn.execute(query)

    for column_name in result.keys(): 
        print(column_name.upper().ljust(20), end='')
    
    print()

    for row in result:
        for field in row:
            print(field.ljust(20), end='')
        
        print()

ID                  FIRSTNAME           LASTNAME            EMAIL               
741109-2058         Alexander           Dahl                alexander.dahl@telia.se
530720-7675         Alvin               Lindholm            alvin.lindholm@gmail.com
820624-3075         Anders              Hansson             anders.hansson@hotmail.com
751123-9724         Anne                Åkerman             anne.akerman@hotmail.com
620925-4245         Annette             Bergfalk            annette.bergfalk@telia.se


In [9]:
import pandas as pd

df = pd.read_sql_query(query, con=engine, index_col="Id")

df

,FirstName,LastName,Email
Id,,,
741109-2058,Alexander,Dahl,alexander.dahl@telia.se
530720-7675,Alvin,Lindholm,alvin.lindholm@gmail.com
820624-3075,Anders,Hansson,anders.hansson@hotmail.com
751123-9724,Anne,Åkerman,anne.akerman@hotmail.com
620925-4245,Annette,Bergfalk,annette.bergfalk@telia.se


create all metadata tabels

In [10]:
from sqlalchemy import MetaData, Table, Column, Integer, String, ForeignKey
metadata_obj = MetaData()

user_table = Table(
    "pythonUsers",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("username", String(30)),
    Column("password", String),
    Column("server_id", ForeignKey('pythonServers.id'), nullable=False)
)

server_table = Table(
    "pythonServers",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("servername", String),
    Column("IP", String(15))
)

for name, table in metadata_obj.tables.items():
    print(f"Table: {name}")

    for column in table.c:
        print(f"{column.name.ljust(15)}{column.type}")

    print()

Table: pythonUsers
id             INTEGER
username       VARCHAR(30)
password       VARCHAR
server_id      INTEGER

Table: pythonServers
id             INTEGER
servername     VARCHAR
IP             VARCHAR(15)



In [11]:
metadata_obj.create_all(engine)

print("Created tables:")

for table in metadata_obj.tables:
    print(table)

Created tables:
pythonUsers
pythonServers
